Install Dependencies

In [ ]:
%pip install -q transformers==4.51.3 datasets==3.5.0 peft==0.15.2 accelerate==1.6.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.12.0 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import torch
import transformers
import datasets
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cu130
Transformers: 4.51.3
Datasets: 3.5.0
PEFT: 0.15.2
Accelerate: 1.6.0
CUDA available: True


Load base Qwen model and tokenizer...

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model onto GPU
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = False

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("Model parameters:", sum(p.numel() for p in model.parameters()))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
GPU: Tesla T4
Model parameters: 494032768


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving train_all.jsonl to train_all.jsonl


In [ ]:
import json

with open("train_all.jsonl", "r", encoding="utf-8") as file:
    examples = [json.loads(line) for line in file]

print("Training examples loaded:", len(examples))

Training examples loaded: 2320


In [ ]:
from datasets import Dataset

MAX_LENGTH = 2048

processed_examples = []
skipped = 0

for example in examples:

    messages = example["messages"]

    prompt_messages = messages[:-1]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True
    )

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("Prompt and conversation tokens do not match.")

    if len(prompt_ids) >= MAX_LENGTH:
        skipped += 1
        continue

    full_ids = full_ids[:MAX_LENGTH]

    labels = [-100] * len(prompt_ids)
    labels += full_ids[len(prompt_ids):]

    processed_examples.append({
        "input_ids": full_ids,
        "labels": labels
    })

train_dataset = Dataset.from_list(processed_examples)

print("Prepared examples:", len(train_dataset))
print("Skipped examples:", skipped)
print("First example token count:", len(train_dataset[0]["input_ids"]))

Prepared examples: 2320
Skipped examples: 0
First example token count: 423


Config. LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [ ]:
import torch

def collate_fn(batch):

    max_length = max(len(item["input_ids"]) for item in batch)

    input_ids = []
    attention_masks = []
    labels = []

    for item in batch:

        padding_length = max_length - len(item["input_ids"])

        input_ids.append(
            item["input_ids"] +
            [tokenizer.pad_token_id] * padding_length
        )

        attention_masks.append(
            [1] * len(item["input_ids"]) +
            [0] * padding_length
        )

        labels.append(
            item["labels"] +
            [-100] * padding_length
        )

    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_masks),
        "labels": torch.tensor(labels)
    }

In [ ]:
from transformers import Trainer, TrainingArguments

OUTPUT_DIR = "/content/reel_rhythm_sft"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,

    fp16=True,

    logging_steps=10,

    save_strategy="epoch",
    save_total_limit=2,

    optim="adamw_torch",
    lr_scheduler_type="cosine",

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    report_to="none",
    label_names=["labels"]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

print("Updated training configuration ready!")

Updated training configuration ready!


Train the model itself..

In [ ]:
trainer.train()

Step,Training Loss
10,2.058300
20,1.857600
30,1.737700
40,1.658700
50,1.703100
60,1.676000
70,1.704600
80,1.660500
90,1.664600
100,1.650300


TrainOutput(global_step=870, training_loss=1.479594185708583, metrics={'train_runtime': 2788.815, 'train_samples_per_second': 2.496, 'train_steps_per_second': 0.312, 'total_flos': 5210560358150400.0, 'train_loss': 1.479594185708583, 'epoch': 3.0})

Save model adaptar and logs...

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

DRIVE_FOLDER = "/content/drive/MyDrive/ReelRhythm"

os.makedirs(DRIVE_FOLDER, exist_ok=True)

print("Backup folder ready:", DRIVE_FOLDER)

Backup folder ready: /content/drive/MyDrive/ReelRhythm


In [ ]:
ADAPTER_DIR = "/content/reel_rhythm_sft_adapter"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("SFT adapter saved locally!")

SFT adapter saved locally!


In [ ]:
import shutil

DRIVE_ADAPTER_DIR = DRIVE_FOLDER + "/sft_adapter"

shutil.copytree(
    ADAPTER_DIR,
    DRIVE_ADAPTER_DIR,
    dirs_exist_ok=True
)

print("SFT adapter successfully backed up to Google Drive!")
print("Location:", DRIVE_ADAPTER_DIR)

SFT adapter successfully backed up to Google Drive!
Location: /content/drive/MyDrive/ReelRhythm/sft_adapter


In [ ]:
import os

print("Backup contents:")

for filename in os.listdir(DRIVE_ADAPTER_DIR):
    filepath = os.path.join(DRIVE_ADAPTER_DIR, filename)

    if os.path.isfile(filepath):
        print(filename, "-", round(os.path.getsize(filepath) / (1024 * 1024), 2), "MB")

Backup contents:
merges.txt - 1.59 MB
tokenizer_config.json - 0.01 MB
tokenizer.json - 10.89 MB
special_tokens_map.json - 0.0 MB
added_tokens.json - 0.0 MB
adapter_config.json - 0.0 MB
README.md - 0.0 MB
vocab.json - 2.65 MB
adapter_model.safetensors - 33.6 MB


In [ ]:
import json

training_results = trainer.state.log_history

with open(DRIVE_FOLDER + "/sft_training_log.json", "w") as file:
    json.dump(training_results, file, indent=4)

print("Training log saved to Google Drive!")

Training log saved to Google Drive!


In [ ]:
import json

log_path = "/content/drive/MyDrive/ReelRhythm/sft_training_log.json"

with open(log_path, "r") as file:
    logs = json.load(file)

for entry in logs:
    if "loss" in entry:
        print(
            "Epoch:", entry.get("epoch"),
            "| Step:", entry.get("step"),
            "| Loss:", entry["loss"]
        )

Epoch: 0.034482758620689655 | Step: 10 | Loss: 2.0583
Epoch: 0.06896551724137931 | Step: 20 | Loss: 1.8576
Epoch: 0.10344827586206896 | Step: 30 | Loss: 1.7377
Epoch: 0.13793103448275862 | Step: 40 | Loss: 1.6587
Epoch: 0.1724137931034483 | Step: 50 | Loss: 1.7031
Epoch: 0.20689655172413793 | Step: 60 | Loss: 1.676
Epoch: 0.2413793103448276 | Step: 70 | Loss: 1.7046
Epoch: 0.27586206896551724 | Step: 80 | Loss: 1.6605
Epoch: 0.3103448275862069 | Step: 90 | Loss: 1.6646
Epoch: 0.3448275862068966 | Step: 100 | Loss: 1.6503
Epoch: 0.3793103448275862 | Step: 110 | Loss: 1.6018
Epoch: 0.41379310344827586 | Step: 120 | Loss: 1.6404
Epoch: 0.4482758620689655 | Step: 130 | Loss: 1.5628
Epoch: 0.4827586206896552 | Step: 140 | Loss: 1.6019
Epoch: 0.5172413793103449 | Step: 150 | Loss: 1.6355
Epoch: 0.5517241379310345 | Step: 160 | Loss: 1.6156
Epoch: 0.5862068965517241 | Step: 170 | Loss: 1.6132
Epoch: 0.6206896551724138 | Step: 180 | Loss: 1.5542
Epoch: 0.6551724137931034 | Step: 190 | Loss: 1.